# CropCare — field-realistic crop disease & pest model

Trains the on-device model that replaces the shipped PlantVillage one.

## Why this exists

The shipped model is stock PlantVillage: 38 classes, 24 of them temperate fruit
a Sri Lankan smallholder will never photograph, **no rice at all**, and exactly
one pest class.

Two measured facts drive everything below:

- A model scoring **99.35%** on PlantVillage's own test split drops to **31.4%**
  on field images.
- A classifier trained on **8 background pixels alone** reaches **49%** accuracy
  on PlantVillage — 19× better than chance. The network is substantially
  reading the *backdrop*, not the leaf.

So PlantVillage is demoted to one source among several, the field datasets carry
the weight, and **PlantDoc is held out of training entirely** as the field test
set. The PlantDoc number at the end is the one that predicts how the app behaves
in an actual field. The validation number is not.

## Before you run

**Settings → Accelerator → GPU T4 x2** (or P100), and **Internet: On**.

Then **Add Input** and attach the four below.

> **Attach data, not notebooks.** The Add Input dialog has separate
> **Competitions**, **Datasets** and **Notebooks** tabs. It is easy to land on
> Notebooks and attach someone's *notebook* named "PlantDoc" — that gives you
> their output, not the images, and nothing will be found. If the Input panel
> groups your attachments under a **NOTEBOOKS** heading, that is what happened.

| Attach from | Search for | Role |
|---|---|---|
| **Competitions** tab | `paddy-disease-classification` | rice, field — **closes the paddy gap** |
| **Competitions** tab | `cassava-leaf-disease-classification` | field survey |
| **Datasets** tab | `plantvillage` or `new-plant-diseases-dataset` | lab, supporting only |
| **Datasets** tab | `plantdoc` | **held out** field test set |

For the two competitions you must open the competition page and accept its
rules first, or the files will not mount even after attaching.

Missing any of them is fine — the notebook reports what it found and trains on
what is there. Missing PlantDoc means you lose the field number, which is the
one worth having.

## 1 · Setup

In [ ]:
import os, sys, json, math, re, random, shutil, collections
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", [g.name for g in gpus] or "NONE — turn on the GPU accelerator")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)

## 2 · Taxonomy

Embedded from `ml/taxonomy.py` in the CropCare repo so this notebook is
self-contained. Edit it there and re-run `python ml/build_notebook.py`, not
here — a class list that has drifted from the app's disease ids is a silent
mismapping, not an error.

In [ ]:
"""
CropCare model taxonomy: the canonical class list, and how each source
dataset's labels map onto it.

This file is the single source of truth. The training notebook builds its label
space from it, the evaluation harness reports against it, and `emit_dart.py`
generates the Dart class list and disease-id map from it. Change a class here
and everything downstream follows.

-----------------------------------------------------------------------------
Why the class list changed
-----------------------------------------------------------------------------
The shipped model is stock PlantVillage: 38 classes, of which 24 are apple,
blueberry, cherry, grape, orange, peach, raspberry, soybean, squash and
strawberry - temperate crops a Sri Lankan smallholder will never photograph.
It has no rice at all, despite rice being the staple crop and despite the app
already seeding `paddy` with disease rows and a translated treatment guideline.
It has exactly one arthropod class, so pests are effectively unsupported.

The taxonomy below drops the temperate fruit entirely and adds rice and
cassava. The count barely changes; the relevance changes completely.

-----------------------------------------------------------------------------
Why the training data changed
-----------------------------------------------------------------------------
PlantVillage is lab photography - detached leaves on uniform backgrounds. A
model scoring 99.35% on its own test split drops to 31.4% on field images, and
a classifier trained on 8 background pixels alone reaches 49% accuracy, which
means the network is substantially reading the backdrop rather than the leaf.

So PlantVillage is kept only as one source among several, and the field
datasets carry the real weight. PlantDoc is held out of training entirely and
used as the field test set - see `HELD_OUT_SOURCES`. A number that is not
measured on unseen field photographs is not worth reporting.
"""


from dataclasses import dataclass, field


# ---------------------------------------------------------------------------
# Canonical classes
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class CropClass:
    """One output class of the model."""

    # Matches the app's `disease` table id convention: {crop}_{condition}.
    id: str
    crop_id: str
    name_en: str

    # 'low' | 'moderate' | 'high' | None. Mirrors disease.severity_default.
    severity: str | None = None

    # True for arthropod damage rather than pathogen infection. The app can
    # word these differently: "pest" and "disease" call for different action,
    # and lumping them under one label was part of why pests went unserved.
    is_pest: bool = False

    healthy: bool = False


def _c(**kw) -> CropClass:
    return CropClass(**kw)


# Ordered. The index of each entry IS the model's output index, so appending
# is safe and reordering is not. `emit_dart.py` relies on this.
CLASSES: list[CropClass] = [
    # -- Rice / paddy ------------------------------------------------------
    # The staple crop, and the app's largest gap: `paddy` is a seeded crop
    # with disease rows and a translated guideline, but the current model
    # cannot predict it at all, so a rice photo returns a confident tomato
    # answer.
    _c(id="paddy_bacterial_leaf_blight", crop_id="paddy",
       name_en="Bacterial Leaf Blight", severity="high"),
    _c(id="paddy_bacterial_leaf_streak", crop_id="paddy",
       name_en="Bacterial Leaf Streak", severity="moderate"),
    _c(id="paddy_bacterial_panicle_blight", crop_id="paddy",
       name_en="Bacterial Panicle Blight", severity="high"),
    _c(id="paddy_blast", crop_id="paddy",
       name_en="Rice Blast", severity="high"),
    _c(id="paddy_brown_spot", crop_id="paddy",
       name_en="Brown Spot", severity="moderate"),
    _c(id="paddy_downy_mildew", crop_id="paddy",
       name_en="Downy Mildew", severity="moderate"),
    _c(id="paddy_tungro", crop_id="paddy",
       name_en="Tungro Virus", severity="high"),
    _c(id="paddy_dead_heart", crop_id="paddy",
       name_en="Stem Borer (Dead Heart)", severity="high", is_pest=True),
    _c(id="paddy_hispa", crop_id="paddy",
       name_en="Rice Hispa", severity="moderate", is_pest=True),
    _c(id="paddy_healthy", crop_id="paddy",
       name_en="Healthy", healthy=True),

    # -- Tomato ------------------------------------------------------------
    _c(id="tomato_bacterial_spot", crop_id="tomato",
       name_en="Bacterial Spot", severity="moderate"),
    _c(id="tomato_early_blight", crop_id="tomato",
       name_en="Early Blight", severity="moderate"),
    _c(id="tomato_late_blight", crop_id="tomato",
       name_en="Late Blight", severity="high"),
    _c(id="tomato_leaf_mold", crop_id="tomato",
       name_en="Leaf Mold", severity="moderate"),
    _c(id="tomato_septoria_leaf_spot", crop_id="tomato",
       name_en="Septoria Leaf Spot", severity="moderate"),
    _c(id="tomato_target_spot", crop_id="tomato",
       name_en="Target Spot", severity="moderate"),
    _c(id="tomato_yellow_leaf_curl_virus", crop_id="tomato",
       name_en="Yellow Leaf Curl Virus", severity="high"),
    _c(id="tomato_mosaic_virus", crop_id="tomato",
       name_en="Mosaic Virus", severity="high"),
    _c(id="tomato_spider_mites", crop_id="tomato",
       name_en="Two-Spotted Spider Mite", severity="moderate", is_pest=True),
    _c(id="tomato_healthy", crop_id="tomato", name_en="Healthy", healthy=True),

    # -- Chili / pepper ----------------------------------------------------
    # PlantVillage's "Pepper,_bell" is the closest available proxy. Noted in
    # the app already (TD-006); it stays a proxy, not a claim of equivalence.
    _c(id="chili_bacterial_spot", crop_id="chili",
       name_en="Bacterial Spot", severity="moderate"),
    _c(id="chili_healthy", crop_id="chili", name_en="Healthy", healthy=True),

    # -- Potato ------------------------------------------------------------
    _c(id="potato_early_blight", crop_id="potato",
       name_en="Early Blight", severity="moderate"),
    _c(id="potato_late_blight", crop_id="potato",
       name_en="Late Blight", severity="high"),
    _c(id="potato_healthy", crop_id="potato", name_en="Healthy", healthy=True),

    # -- Cassava / manioc --------------------------------------------------
    # Widely grown by Sri Lankan smallholders, and the dataset is genuine
    # field survey photography rather than lab plates.
    _c(id="cassava_bacterial_blight", crop_id="cassava",
       name_en="Cassava Bacterial Blight", severity="high"),
    _c(id="cassava_brown_streak", crop_id="cassava",
       name_en="Cassava Brown Streak Disease", severity="high"),
    _c(id="cassava_green_mottle", crop_id="cassava",
       name_en="Cassava Green Mottle", severity="moderate"),
    _c(id="cassava_mosaic", crop_id="cassava",
       name_en="Cassava Mosaic Disease", severity="high"),
    _c(id="cassava_healthy", crop_id="cassava",
       name_en="Healthy", healthy=True),

    # -- Maize -------------------------------------------------------------
    _c(id="corn_gray_leaf_spot", crop_id="corn",
       name_en="Gray Leaf Spot", severity="moderate"),
    _c(id="corn_common_rust", crop_id="corn",
       name_en="Common Rust", severity="moderate"),
    _c(id="corn_northern_leaf_blight", crop_id="corn",
       name_en="Northern Leaf Blight", severity="moderate"),
    _c(id="corn_healthy", crop_id="corn", name_en="Healthy", healthy=True),
]

CLASS_IDS: list[str] = [c.id for c in CLASSES]
CLASS_INDEX: dict[str, int] = {c.id: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

CROPS: list[str] = sorted({c.crop_id for c in CLASSES})


# ---------------------------------------------------------------------------
# Source datasets
# ---------------------------------------------------------------------------
@dataclass
class SourceDataset:
    """A dataset to draw training images from."""

    key: str
    name: str

    # Substrings matched (case-insensitively) against directory names under
    # /kaggle/input, so the notebook works regardless of which mirror of a
    # dataset the user attaches. Exact slugs vary between mirrors and go stale;
    # discovery does not.
    dir_hints: list[str]

    # Maps a source label (a folder name, or a value from a CSV) onto a
    # canonical class id. Anything unmapped is REPORTED, never silently
    # dropped - a quietly discarded third of a dataset is the kind of bug that
    # only shows up as unexplained accuracy loss.
    label_map: dict[str, str] = field(default_factory=dict)

    # Some datasets label via a CSV rather than folder names.
    csv_name: str | None = None
    csv_image_col: str | None = None
    csv_label_col: str | None = None

    # Cassava ships integer labels plus a JSON legend.
    csv_label_is_int: bool = False

    notes: str = ""


PADDY_DOCTOR = SourceDataset(
    key="paddy_doctor",
    name="Paddy Doctor (rice, field)",
    dir_hints=["paddy-disease-classification", "paddy_doctor", "paddy-doctor"],
    csv_name="train.csv",
    csv_image_col="image_id",
    csv_label_col="label",
    notes="Field photography of rice. Closes the app's paddy gap. Two of its "
          "classes (dead_heart, hispa) are insect damage, which gives real "
          "pest coverage for the staple crop without needing IP102.",
    label_map={
        "bacterial_leaf_blight": "paddy_bacterial_leaf_blight",
        "bacterial_leaf_streak": "paddy_bacterial_leaf_streak",
        "bacterial_panicle_blight": "paddy_bacterial_panicle_blight",
        "blast": "paddy_blast",
        "brown_spot": "paddy_brown_spot",
        "downy_mildew": "paddy_downy_mildew",
        "tungro": "paddy_tungro",
        "dead_heart": "paddy_dead_heart",
        "hispa": "paddy_hispa",
        "normal": "paddy_healthy",
    },
)

CASSAVA = SourceDataset(
    key="cassava",
    name="Cassava Leaf Disease (field survey)",
    dir_hints=["cassava-leaf-disease-classification", "cassava"],
    csv_name="train.csv",
    csv_image_col="image_id",
    csv_label_col="label",
    csv_label_is_int=True,
    notes="Collected during a field survey in Uganda; genuinely in-the-wild.",
    label_map={
        "0": "cassava_bacterial_blight",
        "1": "cassava_brown_streak",
        "2": "cassava_green_mottle",
        "3": "cassava_mosaic",
        "4": "cassava_healthy",
    },
)

PLANT_VILLAGE = SourceDataset(
    key="plantvillage",
    name="PlantVillage (lab)",
    dir_hints=["plantvillage", "new-plant-diseases", "plant-village",
               "plant_village"],
    notes="Lab plates on uniform backgrounds. Kept ONLY as extra signal for "
          "classes the field datasets cover thinly. Never the sole source for "
          "a class, and never the test set - see the background-bias note at "
          "the top of this file.",
    label_map={
        "Tomato___Bacterial_spot": "tomato_bacterial_spot",
        "Tomato___Early_blight": "tomato_early_blight",
        "Tomato___Late_blight": "tomato_late_blight",
        "Tomato___Leaf_Mold": "tomato_leaf_mold",
        "Tomato___Septoria_leaf_spot": "tomato_septoria_leaf_spot",
        "Tomato___Target_Spot": "tomato_target_spot",
        "Tomato___Tomato_Yellow_Leaf_Curl_Virus":
            "tomato_yellow_leaf_curl_virus",
        "Tomato___Tomato_mosaic_virus": "tomato_mosaic_virus",
        "Tomato___Spider_mites Two-spotted_spider_mite": "tomato_spider_mites",
        "Tomato___healthy": "tomato_healthy",
        "Pepper,_bell___Bacterial_spot": "chili_bacterial_spot",
        "Pepper,_bell___healthy": "chili_healthy",
        "Potato___Early_blight": "potato_early_blight",
        "Potato___Late_blight": "potato_late_blight",
        "Potato___healthy": "potato_healthy",
        "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot":
            "corn_gray_leaf_spot",
        "Corn_(maize)___Common_rust_": "corn_common_rust",
        "Corn_(maize)___Northern_Leaf_Blight": "corn_northern_leaf_blight",
        "Corn_(maize)___healthy": "corn_healthy",
        # Everything else in PlantVillage - apple, blueberry, cherry, grape,
        # orange, peach, raspberry, soybean, squash, strawberry - is
        # deliberately absent. Those crops are not grown by the users this app
        # is for, and carrying them costs accuracy on the ones that are.
    },
)

PLANT_DOC = SourceDataset(
    key="plantdoc",
    name="PlantDoc (field, HELD OUT)",
    dir_hints=["plantdoc", "plant-doc", "plant_doc"],
    notes="Small, in-the-wild, scraped from the internet. Held out of "
          "training entirely and used as the field test set. This is the "
          "number that actually predicts how the app behaves in a field.",
    label_map={
        # PlantDoc folder naming differs between mirrors, so several spellings
        # map to the same class. Unmapped folders are reported at load time.
        "Tomato leaf bacterial spot": "tomato_bacterial_spot",
        "Tomato Early blight leaf": "tomato_early_blight",
        "Tomato leaf late blight": "tomato_late_blight",
        "Tomato leaf mosaic virus": "tomato_mosaic_virus",
        "Tomato leaf yellow virus": "tomato_yellow_leaf_curl_virus",
        "Tomato Septoria leaf spot": "tomato_septoria_leaf_spot",
        "Tomato mold leaf": "tomato_leaf_mold",
        "Tomato two spotted spider mites leaf": "tomato_spider_mites",
        "Tomato leaf": "tomato_healthy",
        "Bell_pepper leaf spot": "chili_bacterial_spot",
        "Bell_pepper leaf": "chili_healthy",
        "Potato leaf early blight": "potato_early_blight",
        "Potato leaf late blight": "potato_late_blight",
        "Potato leaf": "potato_healthy",
        "Corn Gray leaf spot": "corn_gray_leaf_spot",
        "Corn rust leaf": "corn_common_rust",
        "Corn leaf blight": "corn_northern_leaf_blight",
    },
)

SOURCES: list[SourceDataset] = [PADDY_DOCTOR, CASSAVA, PLANT_VILLAGE, PLANT_DOC]

# Never trained on. Reported separately as the field generalisation number.
HELD_OUT_SOURCES = {"plantdoc"}

# Sources whose images are lab plates. Tracked so the notebook can report what
# fraction of each class's training data is lab rather than field - a class fed
# only by PlantVillage should be treated as unproven no matter what its
# validation accuracy says.
LAB_SOURCES = {"plantvillage"}


def summary() -> str:
    lines = [
        f"{NUM_CLASSES} classes across {len(CROPS)} crops: {', '.join(CROPS)}",
        f"{sum(1 for c in CLASSES if c.is_pest)} pest classes, "
        f"{sum(1 for c in CLASSES if c.healthy)} healthy classes",
        "",
        "Sources:",
    ]
    for s in SOURCES:
        held = " [HELD OUT - field test set]" if s.key in HELD_OUT_SOURCES else ""
        lines.append(f"  - {s.name}{held}: {len(s.label_map)} mapped labels")
    return "\n".join(lines)

print(summary())

## 3 · Config

In [ ]:
IMG_SIZE      = 224          # matches the app's existing preprocessing
BATCH_SIZE    = 64
EPOCHS_HEAD   = 4            # frozen backbone, train the classifier head
EPOCHS_FINE   = 16           # unfreeze the top of the backbone
LR_HEAD       = 1e-3
LR_FINE       = 1e-4
FINE_TUNE_AT  = 100          # unfreeze layers from this index up
VAL_FRACTION  = 0.15

# Fraction of the field dataset (PlantDoc) used for TRAINING rather than test.
#
# It was 0 - the whole of PlantDoc was held out. That produced a test that
# could only fail: 19 of the 34 classes have PlantVillage as their only other
# source, so holding out all of PlantDoc left them with lab plates alone to
# learn from, and then tested them exclusively on field photographs. The first
# run scored 89% on validation and 12.6% on PlantDoc, which was not a
# generalisation gap at all - the two numbers were measuring disjoint sets of
# classes.
#
# Splitting it gives those classes some field exposure while still testing on
# images the model has never seen. The test is now "held-out field images"
# rather than "held-out field dataset" - slightly weaker as evidence, and far
# more useful than a guaranteed zero.
PLANTDOC_TRAIN_FRACTION = 0.5
LABEL_SMOOTH  = 0.05         # the labels are not perfectly clean; do not let
                             # the model become certain about them
DROPOUT       = 0.3

# Caps the lab dataset's contribution. Without this, PlantVillage's sheer
# volume drowns the field data and the model happily relearns the background
# shortcut that makes it useless outdoors.
MAX_LAB_IMAGES_PER_CLASS = 400

OUT_DIR = Path("/kaggle/working")
MODEL_NAME = "cropcare_field_mobilenetv3"

## 4 · Find the attached datasets

Matched by directory name rather than dataset slug, because slugs differ
between mirrors and go stale. Whatever is attached gets used.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
MAX_DEPTH = 3   # datasets, competitions and notebook outputs mount at
                # different depths; searching only the top level misses most
                # of them.

def _norm(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())

def _candidate_dirs():
    """Every directory under /kaggle/input down to MAX_DEPTH."""
    if not INPUT_ROOT.exists():
        return []
    out = []
    for depth in range(1, MAX_DEPTH + 1):
        out.extend(sorted(p for p in INPUT_ROOT.glob("/".join(["*"] * depth))
                          if p.is_dir()))
    return out

def _has_images(folder, limit=1):
    """Cheap check that a directory actually contains images somewhere."""
    n = 0
    for p in folder.rglob("*"):
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            n += 1
            if n >= limit:
                return True
    return False

def find_source_dirs(source):
    """All attached directories plausibly belonging to `source`.

    Matches hints against the path RELATIVE to /kaggle/input, not just the
    directory name, so a dataset nested under a notebook-output mount is still
    found. Parents win over children so the same images are not indexed twice.
    """
    hits = []
    for entry in _candidate_dirs():
        rel = _norm(str(entry.relative_to(INPUT_ROOT)))
        if not any(_norm(h) in rel for h in source.dir_hints):
            continue
        # Shallowest match wins. `_candidate_dirs` yields breadth-first, so any
        # already-accepted hit that contains this one is its ancestor, and
        # indexing both would count every image twice. `is_relative_to` rather
        # than string prefixes: separators differ by platform.
        if any(entry.is_relative_to(h) for h in hits):
            continue
        hits.append(entry)
    return hits

# --- show what is actually mounted ---------------------------------------
print("Mounted under /kaggle/input:")
if not INPUT_ROOT.exists() or not any(INPUT_ROOT.iterdir()):
    print("   (nothing)")
else:
    for top in sorted(p for p in INPUT_ROOT.iterdir() if p.is_dir()):
        print(f"   {top.name}/")
        for child in sorted(p for p in top.iterdir() if p.is_dir())[:12]:
            print(f"      {child.name}/")

print()
FOUND = {}
for src in SOURCES:
    dirs = [d for d in find_source_dirs(src) if _has_images(d)]
    FOUND[src.key] = dirs
    mark = "OK   " if dirs else "MISS "
    held = "  [held out]" if src.key in HELD_OUT_SOURCES else ""
    print(f"{mark} {src.name}{held}")
    for d in dirs:
        print(f"        {d}")

if not any(FOUND[s.key] for s in SOURCES if s.key not in HELD_OUT_SOURCES):
    attached = [p.name for p in INPUT_ROOT.iterdir()] if INPUT_ROOT.exists() else []
    raise SystemExit(
        "No trainable dataset found.\n\n"
        f"What is mounted: {attached or 'nothing'}\n\n"
        "If the Input panel lists these under a NOTEBOOKS heading, you have "
        "attached other people's notebooks instead of the data. Notebook "
        "outputs do not contain these datasets.\n\n"
        "In 'Add Input', use the COMPETITIONS tab for:\n"
        "    paddy-disease-classification\n"
        "    cassava-leaf-disease-classification\n"
        "  (accept each competition's rules on its page first, or it will not "
        "mount)\n"
        "and the DATASETS tab for PlantVillage and PlantDoc."
    )
if not FOUND.get("plantdoc"):
    print("\nWARNING: PlantDoc is not attached, so there will be no field "
          "generalisation number at the end. The validation accuracy alone "
          "will look good and mean very little.")

## 5 · Build one index over every source

Each source is walked into the same shape: `(filepath, class_id, source)`.

Labels that do not map onto the taxonomy are **counted and printed**, never
silently dropped. A quietly discarded third of a dataset shows up later only as
unexplained accuracy loss, which is a miserable thing to debug.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

def _iter_images(folder):
    for p in Path(folder).rglob("*"):
        if p.suffix in IMG_EXT:
            yield p

def load_folder_source(src, roots):
    """Datasets labelled by directory name (PlantVillage, PlantDoc)."""
    rows, unmapped = [], collections.Counter()
    # Normalise once so mirrors that differ only in punctuation still match.
    norm = {re.sub(r"[^a-z0-9]", "", k.lower()): v
            for k, v in src.label_map.items()}
    for root in roots:
        for folder in sorted({p.parent for p in _iter_images(root)}):
            key = re.sub(r"[^a-z0-9]", "", folder.name.lower())
            class_id = norm.get(key)
            if class_id is None:
                unmapped[folder.name] += sum(1 for _ in _iter_images(folder))
                continue
            for img in _iter_images(folder):
                rows.append((str(img), class_id, src.key))
    return rows, unmapped

def load_csv_source(src, roots):
    """Datasets labelled by a CSV (Paddy Doctor, Cassava)."""
    rows, unmapped = [], collections.Counter()
    for root in roots:
        csvs = list(Path(root).rglob(src.csv_name or "train.csv"))
        if not csvs:
            continue
        df = pd.read_csv(csvs[0])
        base = csvs[0].parent
        # Index every image once so we can resolve ids without guessing the
        # directory layout, which differs between competition and mirror.
        by_name = {}
        for p in _iter_images(base):
            by_name.setdefault(p.name, p)
        for _, r in df.iterrows():
            raw = str(r[src.csv_label_col])
            class_id = src.label_map.get(raw)
            if class_id is None:
                unmapped[raw] += 1
                continue
            img_name = str(r[src.csv_image_col])
            p = by_name.get(img_name)
            if p is None:
                continue
            rows.append((str(p), class_id, src.key))
    return rows, unmapped

all_rows = []
for src in SOURCES:
    roots = FOUND[src.key]
    if not roots:
        continue
    if src.csv_name:
        rows, unmapped = load_csv_source(src, roots)
    else:
        rows, unmapped = load_folder_source(src, roots)
    all_rows.extend(rows)
    print(f"{src.name}: {len(rows):,} images")
    if unmapped:
        total = sum(unmapped.values())
        print(f"    {total:,} images under {len(unmapped)} unmapped labels "
              f"(expected for crops outside the taxonomy):")
        for label, n in unmapped.most_common(8):
            print(f"      - {label}: {n:,}")
        if len(unmapped) > 8:
            print(f"      ... and {len(unmapped) - 8} more")

df = pd.DataFrame(all_rows, columns=["path", "class_id", "source"])
if df.empty:
    raise SystemExit("No images matched the taxonomy. Check the attached datasets.")
df["label"] = df["class_id"].map(CLASS_INDEX)
print(f"\nTotal usable: {len(df):,} images across "
      f"{df['class_id'].nunique()} of {NUM_CLASSES} classes")

## 6 · Cap the lab data, then split

Two things happen here and both matter more than they look.

**The lab cap.** PlantVillage has thousands of images per class against Paddy
Doctor's hundreds. Left alone it dominates the gradient and the model relearns
the background shortcut. Capped, it contributes signal without setting the
agenda.

**The split.** Validation comes from the same distribution as training, so it
will look good regardless. PlantDoc is a *different* distribution and never
appears in training, which is why it is the only honest measure here.

In [ ]:
# --- cap the lab source ---------------------------------------------------
capped = []
for (cls, srckey), g in df.groupby(["class_id", "source"]):
    if srckey in LAB_SOURCES and len(g) > MAX_LAB_IMAGES_PER_CLASS:
        g = g.sample(MAX_LAB_IMAGES_PER_CLASS, random_state=SEED)
    capped.append(g)
df = pd.concat(capped, ignore_index=True)

# --- split the field dataset ---------------------------------------------
# Part of PlantDoc joins training so the lab-only classes see real field
# photographs; the rest is never trained on and remains the honest test.
held  = df[df["source"].isin(HELD_OUT_SOURCES)]
other = df[~df["source"].isin(HELD_OUT_SOURCES)]

field_train_parts, field_test_parts = [], []
for cls, g in held.groupby("class_id"):
    g = g.sample(frac=1.0, random_state=SEED)
    n_train = int(len(g) * PLANTDOC_TRAIN_FRACTION)
    field_train_parts.append(g.iloc[:n_train])
    field_test_parts.append(g.iloc[n_train:])

train_pool = pd.concat([other] + field_train_parts, ignore_index=True)
test_df = (pd.concat(field_test_parts, ignore_index=True)
           if field_test_parts else held.reset_index(drop=True))

if len(held):
    print(f"Field data: {sum(len(g) for g in field_train_parts):,} images into "
          f"training, {len(test_df):,} held back for the test.")

# --- stratified train/val -------------------------------------------------
train_parts, val_parts = [], []
for cls, g in train_pool.groupby("class_id"):
    g = g.sample(frac=1.0, random_state=SEED)
    n_val = max(1, int(len(g) * VAL_FRACTION)) if len(g) > 1 else 0
    val_parts.append(g.iloc[:n_val])
    train_parts.append(g.iloc[n_val:])
train_df = pd.concat(train_parts, ignore_index=True).sample(frac=1.0, random_state=SEED)
val_df   = pd.concat(val_parts, ignore_index=True)

print(f"train {len(train_df):,}   val {len(val_df):,}   "
      f"field test (held out) {len(test_df):,}")

# --- composition report ---------------------------------------------------
# A class fed only by lab plates is unproven no matter what validation says.
print("\nPer-class training composition:")
print(f"{'class':<38} {'n':>6}  {'% lab':>6}   note")
lab_only = []
for cls in CLASS_IDS:
    g = train_df[train_df["class_id"] == cls]
    if len(g) == 0:
        print(f"{cls:<38} {0:>6}          NO TRAINING DATA")
        continue
    lab_pct = 100.0 * g["source"].isin(LAB_SOURCES).mean()
    note = ""
    if lab_pct == 100.0:
        note = "lab only — treat as unproven"
        lab_only.append(cls)
    elif len(g) < 100:
        note = "few samples"
    print(f"{cls:<38} {len(g):>6}  {lab_pct:>5.0f}%   {note}")

if lab_only:
    print(f"\n{len(lab_only)} classes have lab-only training data. Their "
          f"validation accuracy will look excellent and should not be believed "
          f"until field images exist for them.")

## 7 · Input pipeline

Augmentation is aimed squarely at the background-bias finding. Aggressive random
cropping and geometric jitter stop the network settling on a fixed leaf position
and a clean backdrop; brightness and contrast jitter stand in for sun, shade and
a cheap phone camera.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def decode(path, label, training):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)   # -> [0, 1]
    if training:
        # Crop before resize: varies scale and framing, which is most of what
        # separates a lab plate from a photo taken over a plant.
        img = tf.image.resize(img, [int(IMG_SIZE * 1.25)] * 2)
        img = tf.image.random_crop(img, [IMG_SIZE, IMG_SIZE, 3])
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, 0.25)
        img = tf.image.random_contrast(img, 0.75, 1.35)
        img = tf.image.random_saturation(img, 0.7, 1.4)
        img = tf.image.random_hue(img, 0.03)
        img = tf.clip_by_value(img, 0.0, 1.0)
    else:
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img, label

def make_ds(frame, training):
    ds = tf.data.Dataset.from_tensor_slices(
        (frame["path"].values, frame["label"].values.astype("int32"))
    )
    if training:
        ds = ds.shuffle(min(len(frame), 8192), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, l: decode(p, l, training), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_df, True)
val_ds   = make_ds(val_df,   False)
test_ds  = make_ds(test_df,  False) if len(test_df) else None

# Class weights: the sources are very unbalanced and the rare classes are not
# the unimportant ones.
counts = train_df["label"].value_counts().to_dict()
total  = sum(counts.values())
class_weight = {
    i: total / (len(counts) * counts[i]) for i in counts
}
print(f"{len(counts)} classes weighted; heaviest "
      f"{max(class_weight.values()):.2f}x, lightest {min(class_weight.values()):.2f}x")

## 8 · Model

`MobileNetV3Large`, chosen for CPU inference on budget hardware — it beats
EfficientNet-Lite on both accuracy and latency, and beats MobileViT on latency
without a GPU delegate.

Two details exist purely so the Flutter app needs no preprocessing changes:

- **Input is `[0,1]`**, and the `[-1,1]` rescale MobileNetV3 expects happens
  *inside* the graph. The app already divides by 255.
- **Output is raw logits, no softmax.** `MlInferenceService` applies softmax
  itself and computes entropy from the distribution; a model that pre-softmaxed
  would double-apply it and quietly flatten every confidence score.

In [ ]:
def build_model():
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input_image")
    # [0,1] -> [-1,1], baked in so the app keeps its existing /255.
    x = tf.keras.layers.Rescaling(scale=2.0, offset=-1.0)(inputs)

    base = tf.keras.applications.MobileNetV3Large(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet",
        include_preprocessing=False,   # we did it above, explicitly
    )
    base.trainable = False

    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(DROPOUT)(x)
    # No activation: raw logits, as the app expects.
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation=None, name="logits")(x)

    return tf.keras.Model(inputs, outputs, name=MODEL_NAME), base

model, base = build_model()

def make_loss(num_classes, smoothing):
    """Cross-entropy with label smoothing, over sparse integer labels.

    `SparseCategoricalCrossentropy` has no `label_smoothing` argument - only
    the one-hot `CategoricalCrossentropy` does - so the labels are one-hotted
    inside the loss and the dataset keeps its cheap integer labels and its
    sparse metrics.

    Smoothing earns its place twice over here: the source labels are not
    perfectly clean, and an app whose entire posture is "do not overstate what
    the model knows" should not be training a maximally confident classifier.
    A softmax that never outputs 0.99 also makes the downstream confidence and
    entropy thresholds easier to set honestly.
    """
    cce = tf.keras.losses.CategoricalCrossentropy(
        from_logits=True, label_smoothing=smoothing
    )

    def loss_fn(y_true, y_pred):
        y_true = tf.one_hot(
            tf.cast(tf.reshape(y_true, [-1]), tf.int32), num_classes
        )
        return cce(y_true, y_pred)

    loss_fn.__name__ = "sparse_cce_label_smoothed"
    return loss_fn

loss = make_loss(NUM_CLASSES, LABEL_SMOOTH)

def compile_model(lr):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss=loss,
        metrics=[
            tf.keras.metrics.SparseCategoricalAccuracy(name="acc"),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3"),
        ],
    )

compile_model(LR_HEAD)
print(f"{model.count_params():,} parameters "
      f"({sum(tf.size(w).numpy() for w in model.trainable_weights):,} trainable)")

## 9 · Train — head first, then fine-tune

In [ ]:
# Weights only, not a full .keras: the model carries a custom loss function,
# which a serialised model would need custom_objects to reload. This file is
# only a crash-recovery artifact anyway - EarlyStopping(restore_best_weights)
# is what actually hands the best weights back at the end.
ckpt = OUT_DIR / f"{MODEL_NAME}.weights.h5"
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        str(ckpt), monitor="val_acc", mode="max",
        save_best_only=True, save_weights_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_acc", mode="max", patience=5,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=1),
]

print("Phase 1 — classifier head, backbone frozen")
hist_head = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
    class_weight=class_weight, callbacks=callbacks, verbose=1,
)

print("\nPhase 2 — fine-tuning the top of the backbone")
base.trainable = True
for layer in base.layers[:FINE_TUNE_AT]:
    layer.trainable = False
# BatchNorm stays frozen: fine-tuning with small batches otherwise wrecks the
# running statistics the pretrained weights depend on.
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

compile_model(LR_FINE)
hist_fine = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_FINE,
    class_weight=class_weight, callbacks=callbacks, verbose=1,
)

## 10 · Evaluate — and the only number that matters

`val_acc` is measured on the same distribution the model trained on, so it will
look good whatever happens. **PlantDoc accuracy is the field number.**

For reference on what a lab-only model does here: PlantVillage-trained networks
report ~99% on their own split and **31.4%** on field images. If the gap below
is anywhere near that wide, the model is not ready and adding field data is the
fix — not more epochs.

In [ ]:
def metrics_of(ds):
    """Named metrics. Keras 3 collapses metrics_names to ['loss',
    'compile_metrics'], so ask for a dict instead of indexing by name."""
    return model.evaluate(ds, verbose=0, return_dict=True)

val_m = metrics_of(val_ds)
print("Validation (same distribution as training - the flattering number):")
for k, v in val_m.items():
    print(f"   {k}: {v:.4f}")

if test_ds is not None:
    test_m = metrics_of(test_ds)
    print("\nField photographs held back from training (the honest number):")
    for k, v in test_m.items():
        print(f"   {k}: {v:.4f}")

    val_acc, test_acc = val_m.get("acc", 0.0), test_m.get("acc", 0.0)
    gap = (val_acc - test_acc) * 100
    print(f"\nLab-to-field gap: {gap:.1f} points")
    if gap > 40:
        print("   SEVERE. Check the per-class table below before touching "
              "hyperparameters - if whole classes read 0.00, the problem is "
              "data coverage, not training.")
    elif gap > 20:
        print("   Large but workable. Try raising PLANTDOC_TRAIN_FRACTION or "
              "lowering MAX_LAB_IMAGES_PER_CLASS.")
    else:
        print("   Reasonable generalisation.")

    # --- per class, split by whether the class had field training data -----
    # An average over classes trained on lab plates and classes trained on
    # field photos is two different measurements added together.
    y_true, y_pred = [], []
    for xb, yb in test_ds:
        y_true.extend(yb.numpy())
        y_pred.extend(np.argmax(model.predict(xb, verbose=0), axis=1))
    y_true, y_pred = np.array(y_true), np.array(y_pred)

    field_backed = set(
        train_df.loc[~train_df["source"].isin(LAB_SOURCES), "class_id"]
    )

    print("\nPer-class accuracy on unseen field images:")
    print(f"{'class':<38} {'n':>4} {'acc':>6}  {'train data':<12} most confused with")
    for i, cls in enumerate(CLASS_IDS):
        m = y_true == i
        if m.sum() == 0:
            continue
        acc = (y_pred[m] == i).mean()
        wrong = y_pred[m][y_pred[m] != i]
        confused = CLASS_IDS[np.bincount(wrong).argmax()] if len(wrong) else "-"
        backing = "field+lab" if cls in field_backed else "LAB ONLY"
        print(f"{cls:<38} {m.sum():>4} {acc:>6.2f}  {backing:<12} {confused}")

    tested = {CLASS_IDS[i] for i in set(y_true)}
    lab_only_tested = sorted(tested - field_backed)
    if lab_only_tested:
        print(f"\n{len(lab_only_tested)} tested classes have NO field training "
              f"data. Their score here measures the lab-to-field gap directly "
              f"and cannot be fixed by training longer:")
        for c in lab_only_tested:
            print(f"   {c}")

    # --- where are the predictions actually going? ------------------------
    # If the model answers one crop for everything, the training set is
    # unbalanced by crop and no amount of epochs will fix it.
    pred_crop = collections.Counter(
        CLASS_IDS[i].split("_")[0] for i in y_pred)
    true_crop = collections.Counter(
        CLASS_IDS[i].split("_")[0] for i in y_true)
    print("\nCrop distribution on the field test set:")
    print(f"{'crop':<12} {'actual':>8} {'predicted':>10}")
    for crop in sorted(set(pred_crop) | set(true_crop)):
        print(f"{crop:<12} {true_crop.get(crop,0):>8} {pred_crop.get(crop,0):>10}")
    print("   A predicted column that piles onto one crop means the training "
          "set is dominated by it.")
else:
    print("\nNo field test set, so there is no generalisation number. "
          "Do not ship on the validation figure alone.")

## 11 · Export TFLite, and verify it still agrees with Keras

Float16 post-training quantisation: roughly half the size, negligible accuracy
cost, and no need for a representative dataset the way int8 would.

The parity check is not ceremony. Conversion silently changing behaviour is the
single most common way one of these projects ships a broken model.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

tflite_path = OUT_DIR / f"{MODEL_NAME}_fp16.tflite"
tflite_path.write_bytes(tflite_model)
size_mb = len(tflite_model) / 1e6
print(f"Wrote {tflite_path.name} — {size_mb:.2f} MB "
      f"(the shipped PlantVillage model is 9.06 MB)")

# --- parity check ---------------------------------------------------------
interp = tf.lite.Interpreter(model_content=tflite_model)
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
print(f"\ninput  {inp['shape']} {inp['dtype'].__name__}")
print(f"output {out['shape']} {out['dtype'].__name__}")
assert tuple(inp["shape"][1:]) == (IMG_SIZE, IMG_SIZE, 3), "unexpected input shape"
assert int(out["shape"][-1]) == NUM_CLASSES, "output width != class count"

check = val_df.sample(min(64, len(val_df)), random_state=SEED)
agree = 0
for _, row in check.iterrows():
    img, _ = decode(tf.constant(row["path"]), tf.constant(0), False)
    batch = tf.expand_dims(img, 0)
    keras_pred = int(np.argmax(model.predict(batch, verbose=0)[0]))
    interp.set_tensor(inp["index"], batch.numpy().astype(inp["dtype"]))
    interp.invoke()
    tfl_pred = int(np.argmax(interp.get_tensor(out["index"])[0]))
    agree += (keras_pred == tfl_pred)

pct = 100.0 * agree / len(check)
print(f"\nKeras/TFLite top-1 agreement on {len(check)} images: {pct:.1f}%")
if pct < 95:
    print("   Below 95% — investigate before shipping. Quantisation should not "
          "change predictions this often.")
else:
    print("   Conversion is faithful.")

## 12 · Generate the Dart the app needs

Emits the class list and the class-index → disease-id map, both ordered to match
the model's output exactly. Copy the printed block into
`lib/data/local/ml/ml_inference_service.dart`.

Also writes `seed_diseases.dart.txt` for the crops and diseases the app does not
yet carry — rice and cassava rows. **Treatment guidance is deliberately not
generated**: fabricated agronomic advice for a real disease is worse than an
empty section, and the repo rules say so.

In [ ]:
lines = []
lines.append("  static const List<String> _classNames = [")
for i, c in enumerate(CLASSES):
    lines.append(f"    '{c.id}',{' ' * max(1, 44 - len(c.id))}// {i}")
lines.append("  ];")
lines.append("")
lines.append("  static const Map<int, String> _classIndexToDiseaseId = {")
for i, c in enumerate(CLASSES):
    lines.append(f"    {i}: '{c.id}',")
lines.append("  };")
lines.append("")
lines.append("  /// Classes that are insect damage rather than infection.")
lines.append("  static const Set<int> _pestClassIndices = {")
pest_idx = [str(i) for i, c in enumerate(CLASSES) if c.is_pest]
lines.append("    " + ", ".join(pest_idx) + ",")
lines.append("  };")

dart = "\n".join(lines)
(OUT_DIR / "ml_class_list.dart.txt").write_text(dart, encoding="utf-8")
print(dart)

# --- rows the app does not have yet --------------------------------------
existing_crops = {"tomato", "chili", "potato", "corn", "paddy"}
seed = []
for crop in sorted({c.crop_id for c in CLASSES}):
    if crop not in existing_crops:
        seed.append(f"// NEW CROP: {crop} - add a CropTableCompanion row")
for c in CLASSES:
    sev = f"const Value('{c.severity}')" if c.severity else "const Value(null)"
    seed.append(
        f"DiseaseTableCompanion.insert(id: '{c.id}', cropId: '{c.crop_id}', "
        f"nameEn: '{c.name_en}', severityDefault: {sev}),"
    )
(OUT_DIR / "seed_diseases.dart.txt").write_text("\n".join(seed), encoding="utf-8")
print(f"\n\nWrote seed_diseases.dart.txt ({len(CLASSES)} disease rows).")
print("name_si / name_ta are NOT generated - those need a native speaker.")

## 13 · Download and wire up

From the **Output** panel, download:

- `cropcare_field_mobilenetv3_fp16.tflite` → `assets/models/`
- `ml_class_list.dart.txt` → paste into `ml_inference_service.dart`
- `seed_diseases.dart.txt` → the new disease rows for `disease_repository_impl.dart`

Then in the app:

1. Update `_modelAsset` and the `assets:` entry in `pubspec.yaml`.
2. Replace `_classNames` and `_classIndexToDiseaseId` with the generated block.
3. Seed the new crops (`cassava`) and disease rows — `crop_repository_impl.dart`
   and `disease_repository_impl.dart`.
4. Re-tune `confidenceThreshold` and `entropyThreshold`. They were fitted to the
   old model's output distribution and do **not** carry over.
5. Re-check `ValidateImageUseCase`'s vegetation-hue gate against rice, which is
   a narrower, greyer leaf than the broadleaf crops it was tuned on.

Still outstanding after this, and neither is a code problem:

- **`name_si` / `name_ta` are empty** for every disease. The columns exist and
  nothing populates them, so disease names render in English in all three
  languages — the single most important string on the result screen.
- **Treatment guidance** for the rice and cassava diseases. Do not invent it.